In [ ]:
import os
import re
import ast
import joblib
import pandas as pd
import numpy as np
from typing import Tuple

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge

# Set dataset path
DATASET_PATH = os.path.join("datasets", "resume_data_for_ranking.csv")
MODEL_EXPORT_PATH = os.path.join("models", "resume_score_model.joblib")

# Load dataset
print(f"Loading dataset from {DATASET_PATH}...")
df = pd.read_csv(DATASET_PATH, low_memory=False)
print(f"Total raw samples: {len(df)}")
df.head(3)

In [ ]:
def clean_text(text) -> str:
    if not isinstance(text, str) or pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s#+.-]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def parse_list_str(val) -> str:
    if not isinstance(val, str) or pd.isna(val):
        return ""
    val = val.strip()
    if val.startswith('[') and val.endswith(']'):
        try:
            parsed = ast.literal_eval(val)
            if isinstance(parsed, list):
                return " ".join([str(item) for item in parsed if item])
        except Exception:
            pass
    return val

def build_resume_text(row) -> str:
    objective = clean_text(row.get('career_objective'))
    skills = clean_text(parse_list_str(row.get('skills')))
    degrees = clean_text(parse_list_str(row.get('degree_names')))
    majors = clean_text(parse_list_str(row.get('major_field_of_studies')))
    positions = clean_text(row.get('positions'))
    resp = clean_text(row.get('responsibilities'))
    return f"{objective} {skills} {degrees} {majors} {positions} {resp}".strip()

def build_job_text(row) -> str:
    title = clean_text(row.get('job_position_name'))
    skills_req = clean_text(parse_list_str(row.get('skills_required')))
    edu_req = clean_text(row.get('educationaL_requirements'))
    exp_req = clean_text(row.get('experiencere_requirement'))
    resp = clean_text(row.get('responsibilities.1'))
    return f"{title} {skills_req} {edu_req} {exp_req} {resp}".strip()

df['full_resume'] = df.apply(build_resume_text, axis=1)
df['full_job'] = df.apply(build_job_text, axis=1)
df['target_score'] = pd.to_numeric(df['matched_score'], errors='coerce') * 100.0

df = df.dropna(subset=['target_score']).reset_index(drop=True)
print(f"Preprocessed & Validated Training Samples: {len(df)}")

In [ ]:

corpus = pd.concat([df['full_resume'], df['full_job']])
vectorizer = TfidfVectorizer(stop_words='english', max_features=1000, ngram_range=(1, 2))
vectorizer.fit(corpus)


res_tfidf = vectorizer.transform(df['full_resume'])
job_tfidf = vectorizer.transform(df['full_job'])

dot_products = np.asarray(res_tfidf.multiply(job_tfidf).sum(axis=1)).flatten()
res_norms = np.sqrt(np.asarray(res_tfidf.multiply(res_tfidf).sum(axis=1)).flatten())
job_norms = np.sqrt(np.asarray(job_tfidf.multiply(job_tfidf).sum(axis=1)).flatten())

denom = res_norms * job_norms
denom[denom == 0] = 1e-6
cosine_sims = dot_products / denom

skill_ratios = []
for idx, row in df.iterrows():
    r_skills = set(clean_text(parse_list_str(row.get('skills'))).split())
    j_skills = set(clean_text(parse_list_str(row.get('skills_required'))).split())
    ratio = len(r_skills.intersection(j_skills)) / float(len(j_skills)) if j_skills else 0.5
    skill_ratios.append(ratio)
skill_ratios = np.array(skill_ratios)

res_lens = df['full_resume'].str.len().values / 1000.0
job_lens = df['full_job'].str.len().values / 1000.0

X = np.column_stack([cosine_sims, skill_ratios, res_lens, job_lens])
y = df['target_score'].values

print(f"Feature Matrix Shape: {X.shape}, Target Vector Shape: {y.shape}")

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    "Ridge Regression": Ridge(alpha=1.0),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1),
    "Gradient Boosting Regressor": GradientBoostingRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
}

best_model = None
best_score = -float('inf')
best_name = ""
results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    results[name] = {"MAE": mae, "RMSE": rmse, "R2": r2}
    print(f"Model: {name:<30} | MAE: {mae:.2f} | RMSE: {rmse:.2f} | R²: {r2:.4f}")
    
    if r2 > best_score:
        best_score = r2
        best_model = model
        best_name = name

print(f"\nWinner Model Selected: {best_name} (R² = {best_score:.4f})")

In [ ]:
class UserResumeScorerPipeline:
    """Serializable pipeline class wrapping TF-IDF vectorizer and Regressor."""
    def __init__(self, vectorizer, regressor):
        self.vectorizer = vectorizer
        self.regressor = regressor

    def predict(self, X_features):
        return self.regressor.predict(X_features)

pipeline = UserResumeScorerPipeline(vectorizer, best_model)

os.makedirs(os.path.dirname(MODEL_EXPORT_PATH), exist_ok=True)
joblib.dump(pipeline, MODEL_EXPORT_PATH)
print(f"Pipeline model serialized to {MODEL_EXPORT_PATH}")

test_sample = np.array([[0.75, 0.80, 0.5, 0.4]])
predicted_score = pipeline.predict(test_sample)[0]
print(f"[Verification Test] Sample Features: {test_sample[0]} => Predicted Match Score: {predicted_score:.2f} / 100")